# Projet Data - Analyse internationale de la satisfaction client

Pipeline de data wrangling et de profilage des enquêtes des cinq filiales.

In [3]:
from pathlib import Path
import importlib
import sys

import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name != "pipeline_v1":
    BASE_DIR = Path(".").resolve()

sys.path.insert(0, str(BASE_DIR / "modules"))

import data_pipeline
import normalization
import standardization
import rag

importlib.reload(data_pipeline)
importlib.reload(normalization)
importlib.reload(standardization)
importlib.reload(rag)

from data_pipeline import (
    build_missing_report,
    build_structure_report,
    calculate_correlations,
    calculate_outlier_rates,
    compare_missing_values,
    correct_data_quality,
    filter_low_correlation,
    impute_numeric_missing,
    load_datasets,
)
from normalization import export_normalized_outputs, normalize_all
from standardization import standardize_all
from rag import answer_with_rag

DATA_DIR = BASE_DIR / "BDD_raw"
STANDARD_DIR = BASE_DIR / "BDD_standard_format"
NORMALIZED_DIR = BASE_DIR / "BDD_normalized_format"
LOW_CORRELATION_THRESHOLD = 0.2

# Sprint 1 : 24/09/2026 - 07/10/2026



## 2.1.1 : Analyse de la structure des datasets

Les fichiers sont lus depuis `BDD_raw`, puis leur structure est comparée.

In [4]:
datasets = load_datasets(DATA_DIR)
structure_report = build_structure_report(datasets)

print(f"Datasets charges : {', '.join(datasets)}")
display(structure_report[[
    "country",
    "rows",
    "columns",
    "numeric_columns",
    "categorical_columns",
    "duplicate_rows",
    "total_missing",
    "missing_rate_global_pct",
    "target_column",
]])

Datasets charges : Allemagne, Canada, Espagne, France, Maroc


,country,rows,columns,numeric_columns,categorical_columns,duplicate_rows,total_missing,missing_rate_global_pct,target_column
0,Allemagne,200,9,5,4,0,50,2.78,None
1,Canada,200,9,5,4,0,50,2.78,overall_satisfaction
2,Espagne,200,9,0,9,0,50,2.78,None
3,France,200,9,5,4,0,50,2.78,satisfaction_globale
4,Maroc,200,5,0,5,0,10,1.00,None


### Profilage des valeurs manquantes

In [5]:
missing_report = build_missing_report(datasets)
display(missing_report.sort_values("missing_rate_pct", ascending=False))

,country,column,dtype,missing_count,row_count,missing_rate_pct
7,Allemagne,kundenservice,float64,13,200,6.5
25,Espagne,servicio_cliente,object,13,200,6.5
34,France,service_client,float64,13,200,6.5
26,Espagne,satisfaccion_global,object,12,200,6.0
4,Allemagne,lieferzeit,float64,11,200,5.5
16,Canada,customer_service,float64,11,200,5.5
24,Espagne,estado_paquete,object,11,200,5.5
17,Canada,overall_satisfaction,float64,11,200,5.5
31,France,delai_livraison,float64,11,200,5.5
15,Canada,parcel_condition,float64,10,200,5.0


# Sprint 2 : 08/10/2026 - 21/10/2026



## 2.2.1 : Filtrage des variables faiblement corrélées

Le filtrage s'applique uniquement aux colonnes numériques. Les variables dont `|r| < 0.2` sont retirées par pays.

In [6]:
correlations = calculate_correlations(datasets)
filtered_datasets, removed_columns = filter_low_correlation(
    datasets,
    correlations,
    threshold=LOW_CORRELATION_THRESHOLD,
)

print(f"Variables retirees : {len(removed_columns)}")
display(correlations)
display(removed_columns)

Variables retirees : 3


,country,target,variable,pearson_r,abs_r,n_valid
0,Canada,overall_satisfaction,delivery_time,0.143718,0.143718,181
1,Canada,overall_satisfaction,delivery_person,0.115156,0.115156,180
2,Canada,overall_satisfaction,parcel_condition,0.154261,0.154261,182
3,Canada,overall_satisfaction,customer_service,0.258725,0.258725,178
4,France,satisfaction_globale,delai_livraison,0.352384,0.352384,181
5,France,satisfaction_globale,relation_livreur,0.201217,0.201217,182
6,France,satisfaction_globale,etat_colis,0.342892,0.342892,184
7,France,satisfaction_globale,service_client,0.367351,0.367351,177


,country,target,variable,pearson_r,abs_r,n_valid,threshold,justification
0,Canada,overall_satisfaction,delivery_time,0.143718,0.143718,181,0.2,Pearson |r| < 0.2: faible correlation avec le ...
1,Canada,overall_satisfaction,delivery_person,0.115156,0.115156,180,0.2,Pearson |r| < 0.2: faible correlation avec le ...
2,Canada,overall_satisfaction,parcel_condition,0.154261,0.154261,182,0.2,Pearson |r| < 0.2: faible correlation avec le ...


## 2.3.1 : Traitement des données manquantes

Taux de valeurs manquantes supérieur à 5 % : médiane. Sinon : moyenne.

In [7]:
imputed_datasets, imputation_log = impute_numeric_missing(filtered_datasets)
missing_comparison = compare_missing_values(filtered_datasets, imputed_datasets)

display(imputation_log)
display(missing_comparison)

,country,column,missing_rate_before_pct,missing_before,method,imputation_value,missing_after
0,Allemagne,lieferzeit,5.5,11,median,4.000000,0
1,Allemagne,zusteller,4.5,9,mean,3.633508,0
2,Allemagne,paket_zustand,3.5,7,mean,3.554404,0
3,Allemagne,kundenservice,6.5,13,median,4.000000,0
4,Allemagne,gesamtzufriedenheit,5.0,10,mean,3.773684,0
5,Canada,customer_service,5.5,11,median,7.000000,0
6,Canada,overall_satisfaction,5.5,11,median,8.000000,0
7,France,delai_livraison,5.5,11,median,4.000000,0
8,France,relation_livreur,4.5,9,mean,3.628272,0
9,France,etat_colis,3.5,7,mean,3.601036,0


,country,missing_before,missing_after,missing_removed
0,Allemagne,50,0,50
1,Canada,22,0,22
2,Espagne,50,50,0
3,France,50,0,50
4,Maroc,10,10,0


## 2.4.1 : Calcul du taux de valeurs aberrantes

Le taux est calculé pour chaque variable numérique avec la méthode de l'écart interquartile (IQR).

In [8]:
outlier_report = calculate_outlier_rates(imputed_datasets)
display(outlier_report.sort_values("outlier_rate_pct", ascending=False))

,country,column,valid_count,outlier_count,outlier_rate_pct,q1,q3,iqr,lower_bound,upper_bound
2,Allemagne,paket_zustand,200,17,8.5,3.0,4.0,1.0,1.5,5.5
10,France,service_client,200,14,7.0,3.0,4.0,1.0,1.5,5.5
9,France,etat_colis,200,13,6.5,3.0,4.0,1.0,1.5,5.5
3,Allemagne,kundenservice,200,13,6.5,3.0,4.0,1.0,1.5,5.5
8,France,relation_livreur,200,10,5.0,3.0,4.0,1.0,1.5,5.5
1,Allemagne,zusteller,200,9,4.5,3.0,4.0,1.0,1.5,5.5
6,Canada,overall_satisfaction,200,5,2.5,6.0,8.0,2.0,3.0,11.0
5,Canada,customer_service,200,3,1.5,6.0,9.0,3.0,1.5,13.5
11,France,satisfaction_globale,200,1,0.5,3.0,5.0,2.0,0.0,8.0
4,Allemagne,gesamtzufriedenheit,200,1,0.5,3.0,5.0,2.0,0.0,8.0


## 2.4.2 : Correction des valeurs aberrantes et des catégories manquantes

Les catégories manquantes sont remplacées par leur mode. Les valeurs numériques aberrantes sont plafonnées aux bornes IQR.

In [9]:
corrected_datasets, correction_log = correct_data_quality(
    imputed_datasets,
    outlier_report,
)

display(correction_log)

,country,column,issue,count_before,count_corrected,method,value
0,Allemagne,zusteller,numeric_outlier,9,9,clip_iqr_bounds,"[1.5, 5.5]"
1,Allemagne,paket_zustand,numeric_outlier,17,17,clip_iqr_bounds,"[1.5, 5.5]"
2,Allemagne,kundenservice,numeric_outlier,13,13,clip_iqr_bounds,"[1.5, 5.5]"
3,Allemagne,gesamtzufriedenheit,numeric_outlier,1,1,clip_iqr_bounds,"[0.0, 8.0]"
4,Canada,customer_service,numeric_outlier,3,3,clip_iqr_bounds,"[1.5, 13.5]"
5,Canada,overall_satisfaction,numeric_outlier,5,5,clip_iqr_bounds,"[3.0, 11.0]"
6,Espagne,plazo_entrega,missing_categorical,9,9,mode,Alto
7,Espagne,relacion_repartidor,missing_categorical,5,5,mode,Medio
8,Espagne,estado_paquete,missing_categorical,11,11,mode,Alto
9,Espagne,servicio_cliente,missing_categorical,13,13,mode,Medio


## Fin du data profiling

Les résultats finaux sont affichés ci-dessous. Aucun fichier de rapport n'est exporté.

In [10]:
print("Fin du data profiling")
print("\nStructure des datasets")
display(structure_report)
print("\nValeurs manquantes")
display(missing_report)
print("\nVariables retirees")
display(removed_columns)
print("\nJournal d'imputation")
display(imputation_log)
print("\nComparaison avant / apres")
display(missing_comparison)
print("\nValeurs aberrantes")
display(outlier_report)
print("\nJournal de correction")
display(correction_log)

Fin du data profiling

Structure des datasets


,country,rows,columns,numeric_columns,categorical_columns,duplicate_rows,total_missing,missing_rate_global_pct,target_column,dtypes
0,Allemagne,200,9,5,4,0,50,2.78,None,umfrage_id: object; umfrage_datum: datetime64[...
1,Canada,200,9,5,4,0,50,2.78,overall_satisfaction,survey_id: object; survey_date: datetime64[ns]...
2,Espagne,200,9,0,9,0,50,2.78,None,id_encuesta: object; fecha_encuesta: object; p...
3,France,200,9,5,4,0,50,2.78,satisfaction_globale,id_enquete: object; date_enquete: datetime64[n...
4,Maroc,200,5,0,5,0,10,1.00,None,id_enquete: object; date_enquete: datetime64[n...



Valeurs manquantes


,country,column,dtype,missing_count,row_count,missing_rate_pct
0,Allemagne,umfrage_id,object,0,200,0.0
1,Allemagne,umfrage_datum,datetime64[ns],0,200,0.0
2,Allemagne,land,object,0,200,0.0
3,Allemagne,produkt_code,object,0,200,0.0
4,Allemagne,lieferzeit,float64,11,200,5.5
5,Allemagne,zusteller,float64,9,200,4.5
6,Allemagne,paket_zustand,float64,7,200,3.5
7,Allemagne,kundenservice,float64,13,200,6.5
8,Allemagne,gesamtzufriedenheit,float64,10,200,5.0
9,Canada,survey_id,object,0,200,0.0



Variables retirees


...

,country,missing_before,missing_after,missing_removed
0,Allemagne,50,0,50
1,Canada,22,0,22
2,Espagne,50,50,0
3,France,50,0,50
4,Maroc,10,10,0



Valeurs aberrantes


,country,column,valid_count,outlier_count,outlier_rate_pct,q1,q3,iqr,lower_bound,upper_bound
0,Allemagne,lieferzeit,200,0,0.0,3.0,5.0,2.0,0.0,8.0
1,Allemagne,zusteller,200,9,4.5,3.0,4.0,1.0,1.5,5.5
2,Allemagne,paket_zustand,200,17,8.5,3.0,4.0,1.0,1.5,5.5
3,Allemagne,kundenservice,200,13,6.5,3.0,4.0,1.0,1.5,5.5
4,Allemagne,gesamtzufriedenheit,200,1,0.5,3.0,5.0,2.0,0.0,8.0
5,Canada,customer_service,200,3,1.5,6.0,9.0,3.0,1.5,13.5
6,Canada,overall_satisfaction,200,5,2.5,6.0,8.0,2.0,3.0,11.0
7,France,delai_livraison,200,0,0.0,3.0,5.0,2.0,0.0,8.0
8,France,relation_livreur,200,10,5.0,3.0,4.0,1.0,1.5,5.5
9,France,etat_colis,200,13,6.5,3.0,4.0,1.0,1.5,5.5



Journal de correction


,country,column,issue,count_before,count_corrected,method,value
0,Allemagne,zusteller,numeric_outlier,9,9,clip_iqr_bounds,"[1.5, 5.5]"
1,Allemagne,paket_zustand,numeric_outlier,17,17,clip_iqr_bounds,"[1.5, 5.5]"
2,Allemagne,kundenservice,numeric_outlier,13,13,clip_iqr_bounds,"[1.5, 5.5]"
3,Allemagne,gesamtzufriedenheit,numeric_outlier,1,1,clip_iqr_bounds,"[0.0, 8.0]"
4,Canada,customer_service,numeric_outlier,3,3,clip_iqr_bounds,"[1.5, 13.5]"
5,Canada,overall_satisfaction,numeric_outlier,5,5,clip_iqr_bounds,"[3.0, 11.0]"
6,Espagne,plazo_entrega,missing_categorical,9,9,mode,Alto
7,Espagne,relacion_repartidor,missing_categorical,5,5,mode,Medio
8,Espagne,estado_paquete,missing_categorical,11,11,mode,Alto
9,Espagne,servicio_cliente,missing_categorical,13,13,mode,Medio


# Sprint 3 : 22/10/2026 - 04/11/2026



## 3.1.1 : Standardisation vers Excel puis JSON

Les cinq datasets traités sont convertis vers un schéma commun, puis écrits dans `BDD_standard_format`.

In [11]:
standardized_datasets, standardization_report = standardize_all(
    corrected_datasets,
    STANDARD_DIR,
)

display(standardization_report)

for country, df in standardized_datasets.items():
    print(f"{country}: {df.shape[0]} lignes, {df.shape[1]} colonnes")

,country,rows,columns,missing_columns,excel_path,json_path,status
0,Allemagne,200,10,commentaire_global,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
1,Canada,200,10,"delai_livraison, relation_livreur, etat_colis,...",c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
2,Espagne,200,10,commentaire_global,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
3,France,200,10,commentaire_global,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
4,Maroc,200,10,"delai_livraison, relation_livreur, etat_colis,...",c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success


Allemagne: 200 lignes, 10 colonnes
Canada: 200 lignes, 10 colonnes
Espagne: 200 lignes, 10 colonnes
France: 200 lignes, 10 colonnes
Maroc: 200 lignes, 10 colonnes


## 3.2.1 : Normalisation des notes sur 5

Les données standardisées et corrigées sont converties vers une échelle commune de satisfaction.

## 3.3.1 : Référentiel produit mondial

Les codes produits locaux sont associés à `STANDARD`, `EXPRESS` ou `PREMIUM`.

In [12]:
normalized_datasets, normalized_base, normalization_report = normalize_all(
    standardized_datasets,
)

assert len(normalized_base) == 1000
assert normalized_base["produit_global"].notna().all()
assert normalized_base["satisfaction_normalisee"].dropna().between(1, 5).all()
assert normalized_base["id_unique"].notna().all()
assert normalized_base["id_unique"].is_unique

normalization_outputs = export_normalized_outputs(
    normalized_datasets,
    normalized_base,
    NORMALIZED_DIR,
)

display(normalization_report)
display(normalized_base.head())

for name, path in normalization_outputs.items():
    print(f"{name}: {path}")

,pays,lignes,colonnes,scores_renseignes,produits_manquants
0,Allemagne,200,7,197,0
1,Canada,200,7,199,0
2,Espagne,200,7,200,0
3,France,200,7,197,0
4,Maroc,200,7,200,0


,id_unique,date_enquete,pays,code_produit,produit_global,satisfaction_normalisee,commentaire_global
0,DE-0001,2026-06-18 00:00:00,Allemagne,PAK_003,PREMIUM,3.773684,NaN
1,DE-0002,2026-09-07 00:00:00,Allemagne,PAK_001,STANDARD,4.000000,NaN
2,DE-0003,2026-04-25 00:00:00,Allemagne,PAK_001,STANDARD,3.000000,NaN
3,DE-0004,2026-06-16 00:00:00,Allemagne,PAK_001,STANDARD,5.000000,NaN
4,DE-0005,2026-01-16 00:00:00,Allemagne,PAK_001,STANDARD,3.773684,NaN


Allemagne_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\allemagne_normalisee.xlsx
Allemagne_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\allemagne_normalisee.json
Canada_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\canada_normalisee.xlsx
Canada_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\canada_normalisee.json
Espagne_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\espagne_normalisee.xlsx
Espagne_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\espagne_normalisee.json
France_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\france_normalisee.xlsx
France_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_

# Sprint 4 : 05/11/2026 - 18/11/2026

## 4.1.1 : Interrogation des donnees avec un RAG

Le systeme retrouve les lignes les plus pertinentes dans la base normalisee, puis transmet uniquement ce contexte a Ollama. Le modele utilise `mistral:latest` par defaut. Ollama doit etre lance localement et le modele doit etre disponible.

In [13]:
rag_data = globals().get("normalized_base")
if rag_data is None:
    normalized_path = NORMALIZED_DIR / "fastcolis_base_normalisee.json"
    if not normalized_path.exists():
        raise FileNotFoundError(
            "Base normalisee introuvable. Executez d'abord les cellules du pipeline "
            "jusqu'a la normalisation."
        )
    rag_data = pd.read_json(normalized_path)

question_rag = input("Posez votre question sur les enquetes FastColis : ")
reponse_rag = answer_with_rag(
    rag_data,
    question_rag,
    model="mistral:latest",
    top_k=8,
 )
print(reponse_rag)

Le produit avec le score global le plus élevé dans le contexte fourni est le "COLIS_PREMIUM", puisque tous les enquêtes concernant ce produit ont un score de satisfaction de 4.0.
